### Importation des bibliothèques

In [1]:
!pip install -U scikit-learn
!pip install tqdm
!pip install ipywidgets

In [2]:
import numpy as np
import scipy.sparse as sp
import scipy.io
import os
from sklearn.decomposition import LatentDirichletAllocation
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import cluster
import re
import os
from sklearn.metrics import normalized_mutual_info_score as nmi_score
from sklearn import metrics
from sklearn.preprocessing import LabelEncoder

### Configuration des chemins

In [13]:
DATA_ROOT = r"C:\Users\Home\Documents\M2\Projet_PPD\data"
OUT_DIR_LDA = r"C:\Users\Home\Documents\M2\Projet_PPD\output\LDA"
JAVA_PATH = r"C:\Program Files\Java\jre1.8.0_51\bin\java.exe"
PALMETTO_JAR = r"C:\Users\Home\Documents\M2\Projet_PPD\palmetto\palmetto-0.1.0-jar-with-dependencies.jar"
WIKI_BD_DIR = r"C:\Users\Home\Documents\M2\Projet_PPD\palmetto\Wikipedia_bd\wikipedia_bd"

if not os.path.exists(OUT_DIR_LDA):
    os.makedirs(OUT_DIR_LDA)

### Fonctions utilitaires de chargement et prétraitement des données

In [25]:
def load_bow_sparse(path):
    X = sp.load_npz(path).astype(np.float32)
    return X.toarray() 

def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_labels(dataset):
    path = os.path.join(DATA_ROOT, dataset, "test_labels.txt")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            # On gère si c'est du texte ou des chiffres
            return [line.strip() for line in f if line.strip()]
    return None

def load_texts(dataset):
    path = os.path.join(DATA_ROOT, dataset, "test_texts.txt")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return [line.strip().split() for line in f if line.strip()]
    return None

### Fonction d'entraînement LDA

In [26]:
def train_one_lda(dataset, K, seed=42, max_iter=50):
    data_dir = os.path.join(DATA_ROOT, dataset)
    out_path = os.path.join(OUT_DIR_LDA, f"{dataset}_LDA_K{K}_seed{seed}.mat")

    # Si le fichier existe déjà, on quitte la fonction 
    if os.path.exists(out_path):
        return out_path

    # Chargement des fichiers du dataset
    X_train = load_bow_sparse(os.path.join(data_dir, "train_bow.npz"))
    X_test  = load_bow_sparse(os.path.join(data_dir, "test_bow.npz"))

    lda = LatentDirichletAllocation(
        n_components=K,           
        max_iter=max_iter,        
        learning_method="batch",
        random_state=seed,
        verbose=0 
    )
    
    lda.fit(X_train) 

    # Préparation des matrices pour export
    beta = lda.components_.astype(np.float32)          
    beta = beta / beta.sum(axis=1, keepdims=True)
    train_theta = lda.transform(X_train).astype(np.float32)  
    test_theta  = lda.transform(X_test).astype(np.float32)   

    scipy.io.savemat(out_path, {"beta": beta, "train_theta": train_theta, "test_theta": test_theta})
    return out_path

### Entraînement des modèles LDA

In [27]:
datasets = ["20NG", "IMDB", "YahooAnswer", "AGNews"] 
ks = [20, 50, 100]
seed = 42

tasks = []
for d in datasets:
    for k in ks:
        filename = f"{d}_LDA_K{k}_seed{seed}.mat"
        full_path = os.path.join(OUT_DIR_LDA, d, filename)
        
        if not os.path.exists(full_path):
            tasks.append((d, k))
        else:
            print(f" Trouvé dans sous-dossier : {d}/{filename}")

if not tasks:
    print("\nTout est prêt !")
else:
    print(f"\nIl reste {len(tasks)} modèles à entraîner.")

 Trouvé dans sous-dossier : 20NG/20NG_LDA_K20_seed42.mat
 Trouvé dans sous-dossier : 20NG/20NG_LDA_K50_seed42.mat
 Trouvé dans sous-dossier : 20NG/20NG_LDA_K100_seed42.mat
 Trouvé dans sous-dossier : IMDB/IMDB_LDA_K20_seed42.mat
 Trouvé dans sous-dossier : IMDB/IMDB_LDA_K50_seed42.mat
 Trouvé dans sous-dossier : IMDB/IMDB_LDA_K100_seed42.mat
 Trouvé dans sous-dossier : YahooAnswer/YahooAnswer_LDA_K20_seed42.mat
 Trouvé dans sous-dossier : YahooAnswer/YahooAnswer_LDA_K50_seed42.mat
 Trouvé dans sous-dossier : YahooAnswer/YahooAnswer_LDA_K100_seed42.mat
 Trouvé dans sous-dossier : AGNews/AGNews_LDA_K20_seed42.mat
 Trouvé dans sous-dossier : AGNews/AGNews_LDA_K50_seed42.mat
 Trouvé dans sous-dossier : AGNews/AGNews_LDA_K100_seed42.mat

Tout est prêt !


### Fonctions pour le calcul des métriques d'évaluation

In [28]:
def get_topic_diversity(beta, topn=15):
    """Calcule la TD sur les 15 mots les plus probables."""
    num_topics = beta.shape[0]
    top_words = []
    for i in range(num_topics):
        idx = np.argsort(beta[i])[-topn:]
        top_words.extend(idx)
    unique_words = set(top_words)
    return len(unique_words) / (num_topics * topn)

def load_palmetto_file(dataset, k, seed=42):
    filename = f"palmetto_CV_{dataset}_LDA_K{k}_seed{seed}.txt"
    path = os.path.join(OUT_DIR_LDA, dataset, filename)
    
    if os.path.exists(path):
        scores = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                match = re.search(r"\d+\s+(\d+,\d+)\s+\[", line)
                if match:
                    score_str = match.group(1).replace(',', '.')
                    scores.append(float(score_str))
        
        if scores:
            return np.mean(scores)
        else:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
                matches = re.findall(r"0,\d{3,}", content)
                scores = [float(m.replace(',', '.')) for m in matches]
                return np.mean(scores) if scores else 0.0
    return 0.0

def get_purity(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

### Récapitulatif des métriques LDA

In [31]:
# --- PARAMÈTRES ---
SAVE_FILE = r"C:\Users\Home\Documents\M2\Projet_PPD\output\LDA\lda_metrics_summary.csv"
force_recalculate = False  

# 1. Vérification de l'existence du fichier
if os.path.exists(SAVE_FILE) and not force_recalculate:
    df_final = pd.read_csv(SAVE_FILE)
    print(f" Résultats chargés depuis {SAVE_FILE}")
else:
    print(" Calcul des métriques en cours...")
    all_results = []
    datasets = ["20NG", "IMDB", "YahooAnswer", "AGNews"]
    ks = [20, 50, 100]

    for d in datasets:
        raw_labels = load_labels(d)
        y_true = LabelEncoder().fit_transform(raw_labels) if raw_labels else None

        for k in ks:
            mat_path = os.path.join(OUT_DIR_LDA, d, f"{d}_LDA_K{k}_seed42.mat")
            
            if os.path.exists(mat_path):
                # Chargement MAT
                data = scipy.io.loadmat(mat_path)
                beta = data['beta']
                theta = data['test_theta']
                y_pred = np.argmax(theta, axis=1)
                
                # Calcul des métriques
                cv_mean = load_palmetto_file(d, k)
                td_score = get_topic_diversity(beta, topn=15)
                nmi_val = nmi_score(y_true, y_pred) if y_true is not None else 0
                purity_val = get_purity(y_true, y_pred) if y_true is not None else 0
                
                all_results.append({
                    "Dataset": d, 
                    "K": k,
                    "CV": round(cv_mean, 3),
                    "TD": round(td_score, 3),
                    "Purity": round(purity_val, 3),
                    "NMI": round(nmi_val, 3)
                })

    df_final = pd.DataFrame(all_results)
    df_final = df_final.sort_values(['Dataset', 'K'])
    df_final.to_csv(SAVE_FILE, index=False)
    print(f" Nouveau calcul terminé et sauvegardé dans {SAVE_FILE}")

display(df_final)

 Résultats chargés depuis C:\Users\Home\Documents\M2\Projet_PPD\output\LDA\lda_metrics_summary.csv


,Dataset,K,CV,TD,Purity,NMI
0,20NG,20,0.384,0.730,0.412,0.432
1,20NG,50,0.380,0.649,0.477,0.414
2,20NG,100,0.374,0.627,0.512,0.414
3,AGNews,20,0.388,0.783,0.740,0.303
4,AGNews,50,0.378,0.804,0.718,0.248
5,AGNews,100,0.377,0.822,0.716,0.228
6,IMDB,20,0.345,0.597,0.727,0.091
7,IMDB,50,0.349,0.663,0.702,0.074
8,IMDB,100,0.345,0.697,0.705,0.068
9,YahooAnswer,20,0.343,0.713,0.447,0.254


In [33]:
paper_lda = {
    "20NG": {
        "K50":  {"CV": 0.385, "TD": 0.655, "Purity": 0.367, "NMI": 0.364},
        "K100": {"CV": 0.387, "TD": 0.622, "Purity": 0.364, "NMI": 0.346}
    },
    "IMDB": {
        "K50":  {"CV": 0.347, "TD": 0.788, "Purity": 0.614, "NMI": 0.041},
        "K100": {"CV": 0.342, "TD": 0.691, "Purity": 0.600, "NMI": 0.037}
    },
    "YahooAnswer": {
        "K50":  {"CV": 0.359, "TD": 0.843, "Purity": 0.288, "NMI": 0.144},
        "K100": {"CV": 0.359, "TD": 0.602, "Purity": 0.297, "NMI": 0.148}
    },
    "AGNews": {
        "K50":  {"CV": 0.364, "TD": 0.864, "Purity": 0.640, "NMI": 0.193},
        "K100": {"CV": 0.349, "TD": 0.696, "Purity": 0.654, "NMI": 0.194}
    }
}

repro_lda = {
    "20NG": {
        "K50":  {"CV": 0.380, "TD": 0.649, "Purity": 0.477, "NMI": 0.414},
        "K100": {"CV": 0.374, "TD": 0.627, "Purity": 0.512, "NMI": 0.414}
    },
    "AGNews": {
        "K50":  {"CV": 0.378, "TD": 0.804, "Purity": 0.718, "NMI": 0.248},
        "K100": {"CV": 0.377, "TD": 0.822, "Purity": 0.716, "NMI": 0.228}
    },
    "IMDB": {
        "K50":  {"CV": 0.349, "TD": 0.663, "Purity": 0.702, "NMI": 0.074},
        "K100": {"CV": 0.345, "TD": 0.697, "Purity": 0.705, "NMI": 0.068}
    },
    "YahooAnswer": {
        "K50":  {"CV": 0.351, "TD": 0.744, "Purity": 0.463, "NMI": 0.248},
        "K100": {"CV": 0.355, "TD": 0.740, "Purity": 0.448, "NMI": 0.232}
    }
}
data = []
metrics = ["CV", "TD", "Purity", "NMI"]

for ds in ["20NG", "IMDB", "YahooAnswer", "AGNews"]:
    for k_key in ["K50", "K100"]:
        row = {"Dataset": ds, "K": int(k_key[1:])}
        for m in metrics:
            p_val = paper_lda[ds][k_key][m]
            r_val = repro_lda[ds][k_key][m]
            row[f"{m}_Papier"] = p_val
            row[f"{m}_Repro"] = r_val
            row[f"{m}_Ecart"] = round(r_val - p_val, 3)
        data.append(row)

df = pd.DataFrame(data).set_index(["Dataset", "K"])

display(df.style
    .format("{:.3f}")
    .set_caption("LDA — Papier vs Repro")
)